In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn (1).csv')

In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [5]:
df.drop(columns=['customerID'],inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape

(7021, 20)

In [9]:
(df['TotalCharges'] == " ").sum()


np.int64(11)

In [10]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


In [11]:
df.isna().sum()

,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0


In [12]:
df = df.dropna(subset=['TotalCharges'])


In [13]:
df['Churn'].value_counts(normalize=True) * 100


,proportion
Churn,
No,73.509272
Yes,26.490728


In [14]:
pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100


Churn,No,Yes
Contract,,
Month-to-month,57.357903,42.642097
One year,88.722826,11.277174
Two year,97.151335,2.848665


In [15]:
df.groupby('Churn')['tenure'].describe()


,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5153.0,37.721133,24.046039,1.0,15.0,38.0,61.0,72.0
Yes,1857.0,18.088853,19.546231,1.0,2.0,10.0,29.0,72.0


In [16]:
X=df.drop(columns=['Churn'])
y=df['Churn']

In [17]:
y.value_counts()


,count
Churn,
No,5153
Yes,1857


In [18]:
y.replace({'Yes':1,'No':0},inplace=True)

/tmp/ipython-input-3269030041.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace({'Yes':1,'No':0},inplace=True)


In [19]:
X_encoded = pd.get_dummies(X, drop_first=True)
print("Before encoding:", X.shape)
print("After encoding:", X_encoded.shape)


Before encoding: (7010, 19)
After encoding: (7010, 30)


In [20]:
X_encoded.isna().sum().sum()

np.int64(0)

In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X_encoded,y,test_size=0.2,random_state=42,stratify=y)


In [22]:
from sklearn.linear_model import LogisticRegression
Lr = LogisticRegression()
Lr.fit(X_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [23]:
y_pred=Lr.predict(X_test)

In [24]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[934  97]
 [174 197]]
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1031
           1       0.67      0.53      0.59       371

    accuracy                           0.81      1402
   macro avg       0.76      0.72      0.73      1402
weighted avg       0.80      0.81      0.80      1402



In [25]:
y_pred_proba = Lr.predict_proba(X_test)[:, 1]
y_pred_30 = (y_pred_proba >= 0.3).astype(int)
confusion_matrix(y_test, y_pred_30)
classification_report(y_test, y_pred_30)

'              precision    recall  f1-score   support\n\n           0       0.89      0.77      0.83      1031\n           1       0.54      0.74      0.62       371\n\n    accuracy                           0.76      1402\n   macro avg       0.72      0.76      0.73      1402\nweighted avg       0.80      0.76      0.77      1402\n'

In [27]:
lr_bal = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

lr_bal.fit(X_train, y_train)
y_pred_bal = lr_bal.predict(X_test)
print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))

[[757 274]
 [ 81 290]]
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1031
           1       0.51      0.78      0.62       371

    accuracy                           0.75      1402
   macro avg       0.71      0.76      0.72      1402
weighted avg       0.80      0.75      0.76      1402



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [28]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    class_weight='balanced'
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


[[751 280]
 [ 82 289]]
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1031
           1       0.51      0.78      0.61       371

    accuracy                           0.74      1402
   macro avg       0.70      0.75      0.71      1402
weighted avg       0.80      0.74      0.76      1402



In [29]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=30,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


[[773 258]
 [ 88 283]]
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1031
           1       0.52      0.76      0.62       371

    accuracy                           0.75      1402
   macro avg       0.71      0.76      0.72      1402
weighted avg       0.80      0.75      0.77      1402



In [31]:
from sklearn.model_selection import GridSearchCV
rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)


In [37]:
param_grid = {
    'n_estimators': [10,50,100, 200,150],
    'max_depth': [5, 8, 10,12],
    'min_samples_leaf': [20, 30, 50,45,35]
}


In [38]:
grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=1
)


In [39]:
grid_rf.fit(X_train, y_train)
grid_rf.best_params_


Fitting 5 folds for each of 100 candidates, totalling 500 fits


{'max_depth': 5, 'min_samples_leaf': 20, 'n_estimators': 150}

In [40]:
best_rf = grid_rf.best_estimator_

y_pred_rf_grid = best_rf.predict(X_test)

print(confusion_matrix(y_test, y_pred_rf_grid))
print(classification_report(y_test, y_pred_rf_grid))


[[751 280]
 [ 77 294]]
              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1031
           1       0.51      0.79      0.62       371

    accuracy                           0.75      1402
   macro avg       0.71      0.76      0.72      1402
weighted avg       0.80      0.75      0.76      1402

